# 5 · Versions, registering a package, the palette

- `@version(n)` marks each version; the current one is the method named `run`. `@upgrade(1, 2)` moves a saved graph on; `@deprecated` marks a version or a node as going away.
- Each module lists its nodes in `NODES`, so importing it registers nothing.
- `cls.describe()` is the record a frontend renders.

In [ ]:
from typing import Annotated

from conductor import Deprecation, OlderVersion, Upgrade, node
from conductor import NodeRegistry, Param, Result, widgets, node


registry = NodeRegistry()


def echo_v1(text: Annotated[str, Param(title="Input", widget=widgets.TextWidget())]) -> Annotated[str, Result(title="Output")]:
    return text

@node("echo", title="Echo", description="Echoes input, with an optional prefix", category="text", older_versions=[OlderVersion(echo_v1, upgrade=Upgrade(), deprecation=Deprecation(header="Use version 2", migration="Version 2 adds a prefix; existing graphs keep working."))])
def echo(
    text: Annotated[str, Param(title="Input", widget=widgets.TextWidget())],
    prefix: Annotated[str, Param(title="Prefix", widget=widgets.TextWidget())] = "",
) -> Annotated[str, Result(title="Output")]:
    return f"{prefix}{text}"


registry = registry.extended_with([echo])

## Versions

A graph pinned at version 1 runs version 1. A deprecation is a notice, not an error.

In [ ]:
print("versions:      ", sorted(echo.versions))
print("current:       ", echo.current)
print("v1 deprecated? ", echo.versions[1].deprecation)
print("v2 deprecated? ", echo.versions[2].deprecation)
print("v1 inputs:     ", [i.name for i in echo.versions[1].interface.inputs])
print("v2 inputs:     ", [i.name for i in echo.versions[2].interface.inputs])

## Registering a package

```python
from myapp.nodes import math, text

for module in (text, math):
    for cls in module.NODES:
        registry.register(cls)
```

Below, a throwaway package in a temporary directory.

In [ ]:
import sys
import tempfile
from pathlib import Path

pkg = Path(tempfile.mkdtemp()) / "demo_nodes"
pkg.mkdir()
(pkg / "__init__.py").write_text("")
(pkg / "shout.py").write_text('''
from typing import Annotated

from conductor import Param, node, widgets



@node("shout", title="Shout", description="Uppercases and adds a bang", category="text")
def shout(text: Annotated[str, Param(title="Input", widget=widgets.TextWidget())]) -> str:
    return text.upper() + "!"


NODES = (shout,)
''')

sys.path.insert(0, str(pkg.parent))
from demo_nodes import shout  # noqa: E402

registry = registry.extended_with(shout.NODES)
print("registered:", list(registry))

## The palette

`describe()` dumps through pydantic: a `dtype` as `{"id": ...}`, a widget as its `kind` and options.

In [ ]:
import json

from conductor import NodeDescription, node
from pydantic import TypeAdapter

descriptions = [cls.describe() for cls in registry.nodes]
palette = TypeAdapter(list[NodeDescription]).dump_python(descriptions, mode="json")
print(json.dumps(palette[0], indent=2))

## Summary

In [ ]:
print(f"Total nodes: {len(palette)}\n")
for node in palette:
    print(f"  {node['id']}: {node['title']} (current: v{node['current']})")
    for number, ver in node["versions"].items():
        status = "DEPRECATED" if ver["deprecation"] else "current" if int(number) == node["current"] else "older"
        print(f"    v{number} ({status})")
        for inp in ver["inputs"]:
            opt = " (optional)" if not inp["required"] else ""
            print(f"      in:  {inp['name']}: {inp['dtype']['id']} [{inp['widget']['kind']}]{opt}")
        for out in ver["outputs"]:
            print(f"      out: {out['name']}: {out['dtype']['id']}")
    print()